# 02 Demand distribution: exploration

**Question.** How is demand distributed per station $s$, category $k$ and weekday, and what does that say about
the demand to come $D_{s,k,d}$ in the 14 horizon days?

Two measures of demand, both from the history (`completed` + `open` rentals = every pickup in the 8 weeks).
Future bookings (`booked`, picked up on or after day 1) are not used anywhere in this notebook.
- **cars on rent** $T_{s,k,t}$: rentals picked up at $s$ that occupy a car on date $t$ (the quantity R7 uses);
- **pickups** $P_{s,k,t}$: rentals starting on date $t$ (the arrival process of route A in [[Project Plan]]).

**Truncation.** The history only holds rentals picked up in the window. A rental can last up to 10 days, so $T$ is
complete only from the 10th day of the window on. Earlier dates are dropped for $T$; pickups are complete everywhere.

Sections: 1–2 the asked-for distributions per weekday × station × category; 3 rental length (shape, pooling,
independence) and lead time; 4 dispersion (Poisson or not, against a simulated null); 5 weekday and trend;
6 correlations (upgrades, scenario generation); 7 empirical demand to come from pseudo-snapshots.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

# Work from Project_4/notebooks or from Project_4 itself.
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
from fleetkit import load_instance  # noqa: E402
from fleetopt import level0  # noqa: E402

inst = load_instance(data_dir=ROOT / "data")
CATS = level0.categories_in_order(inst)                      # economy < compact < midsize < premium
CAT_COLOR = dict(zip(CATS, ["#2a78d6", "#eb6834", "#1baf7a", "#eda100"]))
TYPE_COLOR = {"airport": "#2a78d6", "downtown": "#eb6834"}
ST = inst.stations.set_index("station_id")
STATIONS = ST.index.tolist()
ST_LABEL = {s: f"{s} {ST.loc[s, 'station_name']}\n({ST.loc[s, 'station_type']})" for s in STATIONS}
WEEKDAYS = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
DAY1 = level0.horizon_dates(inst)[0]                         # 2027-03-01, a Monday

plt.rcParams.update({
    "figure.dpi": 90, "font.size": 9, "axes.titlesize": 9, "axes.labelsize": 8,
    "xtick.labelsize": 7, "ytick.labelsize": 7, "axes.spines.top": False,
    "axes.spines.right": False, "axes.grid": True, "grid.color": "#e4e3df", "grid.linewidth": 0.6,
    "axes.axisbelow": True, "axes.edgecolor": "#9a9993",
})
pd.set_option("display.width", 200, "display.max_columns", 40)

# One row per reservation, with the derived columns we use everywhere.
r = inst.reservations.copy()
for c in ["pickup_date", "return_date", "booking_date"]:
    r[c] = pd.to_datetime(r[c])
r["length"] = (r["return_date"] - r["pickup_date"]).dt.days + 1     # car-days, both ends counted
r["lead"] = (r["pickup_date"] - r["booking_date"]).dt.days           # days booked ahead
r["rev_per_day"] = r["revenue"] / r["length"]
r["weekday"] = pd.Categorical(r["pickup_date"].dt.day_name().str[:3], WEEKDAYS, ordered=True)
r["week"] = (r["pickup_date"] - r["pickup_date"].min()).dt.days // 7 + 1
r["station_type"] = r["pickup_station_id"].map(ST["station_type"])
r["category"] = pd.Categorical(r["category"], CATS, ordered=True)
# History only: future bookings (status "booked", picked up on or after day 1) are left out of all exploration.
# `open` rentals stay in: they were picked up before day 1 and are part of the observed past demand.
r = r[r["status"] != "booked"].reset_index(drop=True)
print(r["status"].value_counts().to_string())
print("pickup dates:", r["pickup_date"].min().date(), "...", r["pickup_date"].max().date())


def grid_axes(nrows, ncols, w=3.6, h=2.0, **kw):
    """Small-multiples grid: stations as rows, categories as columns."""
    fig, axes = plt.subplots(nrows, ncols, figsize=(w * ncols, h * nrows), squeeze=False, **kw)
    return fig, axes


def label_grid(axes, rows=STATIONS, cols=CATS):
    for j, k in enumerate(cols):
        axes[0, j].set_title(k, fontsize=11, fontweight="bold", color=CAT_COLOR.get(k, "k"))
    for i, s in enumerate(rows):
        axes[i, 0].set_ylabel(ST_LABEL.get(s, s), fontsize=8, rotation=0, ha="right", va="center", labelpad=8)

In [ ]:
H0 = r.loc[r["status"] == "completed", "pickup_date"].min()        # first day of the history window
H1 = DAY1 - pd.Timedelta(days=1)                                    # last day before day 1
hw = r[r["status"].isin(["completed", "open"])].reset_index(drop=True)   # every pickup in the window
MAXLEN = int(hw["length"].max())
FULL_FROM = H0 + pd.Timedelta(days=MAXLEN - 1)                      # cars on rent fully observed from here
dates = pd.date_range(H0, H1)
print(f"window {H0.date()} .. {H1.date()} ({len(dates)} days), max length {MAXLEN}, "
      f"cars on rent complete from {FULL_FROM.date()} ({(H1 - FULL_FROM).days + 1} days)")


def explode_days(df: pd.DataFrame) -> pd.DataFrame:
    """One row per (rental, occupied date)."""
    rep = df.loc[df.index.repeat(df["length"])].copy()
    rep["date"] = rep["pickup_date"] + pd.to_timedelta(np.concatenate([np.arange(n) for n in df["length"]]), unit="D")
    return rep


def on_grid(counts: pd.Series, dates: pd.DatetimeIndex, name: str) -> pd.DataFrame:
    idx = pd.MultiIndex.from_product([STATIONS, CATS, dates], names=["station_id", "category", "date"])
    out = counts.reindex(idx, fill_value=0).rename(name).reset_index()
    out["weekday"] = pd.Categorical(out["date"].dt.day_name().str[:3], WEEKDAYS, ordered=True)
    out["week"] = (out["date"] - H0).dt.days // 7 + 1
    out["station_type"] = out["station_id"].map(ST["station_type"])
    return out


occ_rows = explode_days(hw)
occ_rows = occ_rows[occ_rows["date"] <= H1]
T = on_grid(occ_rows.groupby(["pickup_station_id", "category", "date"], observed=True).size()
                    .rename_axis(["station_id", "category", "date"]), dates, "on_rent")
T = T[T["date"] >= FULL_FROM].reset_index(drop=True)
P = on_grid(hw.groupby(["pickup_station_id", "category", "pickup_date"], observed=True).size()
              .rename_axis(["station_id", "category", "date"]), dates, "pickups")
print(f"T: {len(T)} cells ({T['date'].nunique()} dates),  P: {len(P)} cells ({P['date'].nunique()} dates)")

## 1. Cars on rent per weekday, per station × category

Each panel: one box per weekday over the fully observed history dates (6–7 values each), every value as a dot,
the mean as a dark tick. The y-axis starts at 0 so the panels show the level as well as the spread.

In [ ]:
def weekday_grid(df, col, title):
    fig, axes = grid_axes(len(STATIONS), len(CATS), w=3.7, h=1.8)
    rng = np.random.default_rng(0)
    for i, s in enumerate(STATIONS):
        for j, k in enumerate(CATS):
            ax = axes[i, j]
            g = df[(df["station_id"] == s) & (df["category"] == k)]
            vals = [g.loc[g["weekday"] == w, col].to_numpy() for w in WEEKDAYS]
            ax.boxplot(vals, positions=range(7), widths=0.6, showfliers=False, whis=(0, 100),
                       medianprops={"color": CAT_COLOR[k], "lw": 1.5},
                       boxprops={"color": "#9a9993"}, whiskerprops={"color": "#9a9993"}, capprops={"color": "#9a9993"})
            for x, v in enumerate(vals):
                ax.scatter(x + rng.uniform(-0.15, 0.15, len(v)), v, s=7, color=CAT_COLOR[k], alpha=0.7, zorder=3)
                ax.plot([x - 0.3, x + 0.3], [v.mean()] * 2, color="#0b0b0b", lw=1.2, zorder=4)
            ax.set_ylim(bottom=0)
            ax.set_xticks(range(7), WEEKDAYS if i == len(STATIONS) - 1 else [""] * 7)
    label_grid(axes)
    fig.suptitle(title, y=1.0, fontsize=12)
    fig.tight_layout()
    plt.show()

weekday_grid(T, "on_rent", "Cars on rent per day (history), by pickup station × category × weekday")

### Table: mean and coefficient of variation of cars on rent per (station, category, weekday)

In [ ]:
tab = T.groupby(["station_id", "category", "weekday"], observed=True)["on_rent"].agg(["mean", "std"])
tab["cv"] = tab["std"] / tab["mean"]
print("mean cars on rent")
display(tab["mean"].unstack("weekday").round(1))
print("coefficient of variation")
display(tab["cv"].unstack("weekday").round(2))

## 2. Pickups per weekday, per station × category

Same layout for the arrival process (8 values per weekday).

In [ ]:
weekday_grid(P, "pickups", "Pickups per day (history), by station × category × weekday")

## 3. Rental length (and lead time)

**Why before the dispersion section.** Section 4 compares cars on rent with a Poisson model. The result behind it
($M_t/G/\infty$ on a day grid): if pickups on day $p$ are $N_p\sim\mathrm{Poisson}(\lambda_p)$, independent across days,
and every rental draws its length independently from a pmf $\pi_p$, then
$$T_d=\sum_{p}\sum_{i\le N_p}\mathbf 1\{L_i\ge d-p+1\}\ \sim\ \mathrm{Poisson}\Big(\sum_p\lambda_p\,S_p(d-p+1)\Big),\qquad S_p(l)=P_{\pi_p}(L\ge l),$$
by Poisson thinning and superposition. This holds for **any** length distribution, which may even differ by pickup day.
The length pmf sets the *mean* of $T$ and its autocorrelation over days, not whether it is Poisson.

What the result does need, and what we check here:
1. **Shape and pooling:** what $\pi$ looks like, and whether one pmf per station type × category (or even one overall) is enough.
2. **Independence:** lengths independent of the pickup weekday, of the history week, of how busy the day is,
   and of each other within a day (no day-level clustering).
3. Lead time (for the committed vs. to-come split in section 7).

Data: `completed` + `open` rentals (every pickup in the window, all lengths known).

In [ ]:
LS = np.arange(1, MAXLEN + 1)

def fit_trunc_geom(counts: np.ndarray) -> float:
    """MLE of q for the geometric pmf truncated at MAXLEN, pi(l) ∝ q^(l-1), l = 1..MAXLEN (grid search)."""
    qs = np.linspace(0.01, 0.99, 981)[:, None]
    ll = (counts * (LS - 1) * np.log(qs)).sum(1) - counts.sum() * np.log((qs ** (LS - 1)).sum(1))
    return float(qs[ll.argmax(), 0])

cnt = (hw.groupby(["station_type", "category"], observed=True)["length"].value_counts()
         .unstack(fill_value=0).reindex(columns=LS, fill_value=0))
fig, axes = plt.subplots(1, 2, figsize=(14, 3.8), sharey=True)
for ax, t in zip(axes, ["airport", "downtown"]):
    for k in CATS:
        c = cnt.loc[(t, k)].to_numpy()
        ax.plot(LS, c / c.sum(), marker="o", ms=4, lw=1.5, color=CAT_COLOR[k], label=f"{k} (n={c.sum()})")
    c = cnt.loc[t].sum().to_numpy()
    q = fit_trunc_geom(c)
    geo = q ** (LS - 1) / (q ** (LS - 1)).sum()
    ax.plot(LS, geo, color="#0b0b0b", lw=1.2, ls="--", label=f"truncated geometric, q = {q:.3f}")
    ax.set_yscale("log"); ax.set_xticks(LS); ax.set_xlabel("rental length (car-days)")
    ax.set_title(f"{t}: length pmf (log scale)"); ax.legend(fontsize=7)
fig.tight_layout(); plt.show()
print("length per station type x category:")
display(hw.groupby(["station_type", "category"], observed=True)["length"].agg(["mean", "std", "size"]).unstack(0).round(2))

**Can we pool?** χ² homogeneity tests of the length distribution across groups (lengths 8–10 merged into "8+" so
every expected count is ≥ 5; p-values from the Wilson–Hilferty normal approximation). Large p = no evidence that the
groups differ, so pooling them is fine.

In [ ]:
from math import erfc, sqrt

def chi2_sf(x: float, df: int) -> float:
    """Upper tail of the chi-square distribution (Wilson-Hilferty approximation, good for df >= 5)."""
    z = ((x / df) ** (1 / 3) - (1 - 2 / (9 * df))) / sqrt(2 / (9 * df))
    return 0.5 * erfc(z / sqrt(2))

def chi2_homogeneity(tab) -> tuple[float, int]:
    tab = np.asarray(tab, float)
    tab = tab[tab.sum(1) > 0][:, tab.sum(0) > 0]
    e = tab.sum(1, keepdims=True) * tab.sum(0, keepdims=True) / tab.sum()
    return float(((tab - e) ** 2 / e).sum()), (tab.shape[0] - 1) * (tab.shape[1] - 1)

hw["len_band"] = hw["length"].clip(upper=8)
def counts(keys):
    return hw.groupby(keys, observed=True)["len_band"].value_counts().unstack(fill_value=0)

tests = {
    "all 40 station x category cells alike": [counts(["pickup_station_id", "category"])],
    "the 8 station type x category groups alike": [counts(["station_type", "category"])],
    "stations alike, within station type x category": [g for _, g in counts(["station_type", "category", "pickup_station_id"]).groupby(level=[0, 1])],
    "categories alike, within station": [g for _, g in counts(["pickup_station_id", "category"]).groupby(level=0)],
    "airport vs downtown (pooled over categories)": [counts(["station_type"])],
    "categories (pooled over stations)": [counts(["category"])],
}
rows = []
for name, tabs in tests.items():
    parts = [chi2_homogeneity(t) for t in tabs]
    x, df = sum(p[0] for p in parts), sum(p[1] for p in parts)
    rows.append({"H0: length pmf is the same for ...": name, "chi2": round(x, 1), "df": df, "p": round(chi2_sf(x, df), 4)})
display(pd.DataFrame(rows))

**Independence checks.** Mean length (with 95 % CI) by pickup weekday and by history week; mean length against how busy
the pickup day was (pickups / the cell's weekday mean); and day-level clustering:
$z = (\bar L_{\text{day}} - \bar L_{s,k}) / (\mathrm{sd}_{s,k}/\sqrt{n_{\text{day}}})$ per (station, category, pickup date).
Lengths independent within a day ⇒ $\mathrm{Var}(z) \approx 1$.

In [ ]:
def mean_ci(v: pd.Series) -> pd.Series:
    return pd.Series({"m": v.mean(), "h": 1.96 * v.std() / np.sqrt(len(v)), "n": len(v)})

day = hw.groupby(["pickup_station_id", "category", "pickup_date"], observed=True)["length"].agg(["mean", "size"]).reset_index()
day["weekday"] = day["pickup_date"].dt.day_name().str[:3]
day["busy"] = day["size"] / day.groupby(["pickup_station_id", "category", "weekday"], observed=True)["size"].transform("mean")
cell = hw.groupby(["pickup_station_id", "category"], observed=True)["length"].agg(mu="mean", sd="std")
day = day.join(cell, on=["pickup_station_id", "category"])
day["dev"] = day["mean"] - day["mu"]
day["z"] = day["dev"] / (day["sd"] / np.sqrt(day["size"]))

fig, axes = plt.subplots(1, 4, figsize=(19, 3.6))
for ax, col, title in [(axes[0], "weekday", "mean length by pickup weekday"), (axes[1], "week", "mean length by history week")]:
    for t_i, t in enumerate(["airport", "downtown"]):
        e = hw[hw["station_type"] == t].groupby(col, observed=True)["length"].apply(mean_ci).unstack()
        xs = np.arange(len(e)) + (t_i - 0.5) * 0.2
        ax.errorbar(xs, e["m"], yerr=e["h"], fmt="o", ms=4, color=TYPE_COLOR[t], ecolor=TYPE_COLOR[t], capsize=2, label=t)
        ax.set_xticks(np.arange(len(e)), [str(v) for v in e.index])
    ax.set_title(title); ax.legend(fontsize=7)

ax = axes[2]
day["busy_bin"] = pd.qcut(day["busy"], 8)
b = day.groupby("busy_bin", observed=True).apply(
    lambda g: pd.Series({"x": g["busy"].mean(), "m": np.average(g["dev"], weights=g["size"]),
                         "h": 1.96 * np.sqrt(np.average(g["dev"] ** 2 * g["size"], weights=g["size"]) / g["size"].sum())}),
    include_groups=False)
ax.errorbar(b["x"], b["m"], yerr=b["h"], fmt="o", ms=5, color="#2a78d6", ecolor="#9a9993", capsize=2)
ax.axhline(0, color="#52514e", lw=0.8)
rho_busy = np.corrcoef(day["busy"], day["dev"])[0, 1]
ax.set_xlabel("pickups / cell weekday mean (8 bins)"); ax.set_ylabel("day mean length − cell mean")
ax.set_title(f"length vs. how busy the day was: ρ = {rho_busy:+.3f}")

ax = axes[3]
zz_ = day.loc[day["size"] >= 2, "z"]
ax.hist(zz_, bins=60, density=True, color="#2a78d6", edgecolor="white", linewidth=0.3)
xg = np.linspace(-4, 4, 200); ax.plot(xg, np.exp(-xg ** 2 / 2) / np.sqrt(2 * np.pi), color="#0b0b0b", lw=1.2)
ax.set_xlim(-5, 5); ax.set_title(f"day-level clustering: Var(z) = {zz_.var():.2f} (independent: 1)")
fig.tight_layout(); plt.show()

def spearman(a: pd.Series, b: pd.Series) -> float:
    """Rank correlation without scipy: Pearson on average ranks."""
    return a.rank().corr(b.rank())

print(f"Spearman(length, lead): all {spearman(hw['length'], hw['lead']):+.3f}, "
      + ", ".join(f"{t} {spearman(g['length'], g['lead']):+.3f}" for t, g in hw.groupby("station_type")))

**Lead time.** CDF per station type × category, and mean lead vs. mean length per station.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
ax = axes[0]
for t, ls in [("airport", "-"), ("downtown", "--")]:
    for k in CATS:
        e = np.sort(hw[(hw["station_type"] == t) & (hw["category"] == k)]["lead"].to_numpy())
        ax.step(e, np.arange(1, len(e) + 1) / len(e), where="post", ls=ls, lw=1.5, color=CAT_COLOR[k], label=f"{k}, {t}")
ax.set_xlim(0, 30); ax.set_xlabel("lead time (days)"); ax.set_title("lead time CDF  P(ℓ ≤ x)  (solid airport, dashed downtown)")
ax.legend(fontsize=6, ncol=2)
ax = axes[1]
lt = hw.groupby(["pickup_station_id"], observed=True)[["length", "lead"]].mean()
ax.scatter(lt["length"], lt["lead"], s=30, c=[TYPE_COLOR[ST.loc[s, "station_type"]] for s in lt.index])
for s, row in lt.iterrows():
    ax.annotate(s, (row["length"], row["lead"]), fontsize=7, xytext=(3, 3), textcoords="offset points")
ax.set_xlabel("mean length"); ax.set_ylabel("mean lead"); ax.set_title("per station (blue airport, orange downtown)")
fig.tight_layout(); plt.show()
print("share of rentals with lead <= x days, by station type x category")
display(hw.groupby(["station_type", "category"], observed=True)["lead"]
          .apply(lambda v: pd.Series({f"<= {x}": (v <= x).mean() for x in [0, 1, 3, 7, 14]})).unstack().round(3))

## 4. Dispersion: Poisson or overdispersed?

By section 3, if pickups are Poisson and lengths independent, cars on rent are Poisson too, whatever the length pmf.
So both $P$ and $T$ should have variance ≈ mean per (station, category, weekday). With only 6–8 values per cell, a
single cell's Var/mean is very noisy (under Poisson with $n=8$ it is $\chi^2_7/7$, sd ≈ 0.53), so we look at pooled
statistics: the median and mean of Var/mean over cells, and the pooled excess $\phi = \sum \mathrm{Var} / \sum \mu - 1$
(0 under Poisson; under NegBin with shape $r$ it is $\sum\mu^2 / (r\sum\mu)$).

In [ ]:
def dispersion(df, col):
    g = df.groupby(["station_id", "category", "weekday"], observed=True)[col].agg(["mean", "var", "count"])
    g = g[g["mean"] > 0]
    g["ratio"] = g["var"] / g["mean"]
    stats = {"median": g["ratio"].median(), "mean": g["ratio"].mean(), "phi": g["var"].sum() / g["mean"].sum() - 1}
    excess = (g["var"] - g["mean"]).sum()
    stats["r_hat"] = (g["mean"] ** 2).sum() / excess if excess > 0 else np.inf
    return g, stats

obs = {}
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (key, df, col, lab) in zip(axes[:2], [("P", P, "pickups", "pickups"), ("T", T, "on_rent", "cars on rent")]):
    g, st = dispersion(df, col)
    obs[key] = st
    for k in CATS:
        e = g.xs(k, level="category")
        ax.scatter(e["mean"], e["var"], s=10, alpha=0.7, color=CAT_COLOR[k], label=k)
    lim = [0.3, g[["mean", "var"]].max().max() * 1.3]
    ax.plot(lim, lim, color="#0b0b0b", lw=1, label="Var = mean (Poisson)")
    ax.set_xscale("log"); ax.set_yscale("log")
    ax.set_xlabel("mean per (s,k,weekday)"); ax.set_ylabel("variance")
    ax.set_title(f"{lab}: median Var/mean = {st['median']:.2f}, NegBin r̂ = {st['r_hat']:.0f}")
    print(f"{lab:13s}: median Var/mean {st['median']:.2f}, mean {st['mean']:.2f}, pooled excess φ {st['phi']:+.3f}, NegBin r̂ {st['r_hat']:.1f}")
axes[0].legend(fontsize=7)
g, _ = dispersion(P, "pickups")
axes[2].hist(g["ratio"], bins=40, density=True, color="#2a78d6", edgecolor="white", linewidth=0.3, label="pickups")
from math import gamma
x = np.linspace(0.01, 4, 300); n = 8
dens = (n - 1) * ((n - 1) * x) ** ((n - 1) / 2 - 1) * np.exp(-(n - 1) * x / 2) / (2 ** ((n - 1) / 2) * gamma((n - 1) / 2))
axes[2].plot(x, dens, color="#0b0b0b", lw=1.2, label="Poisson: χ²₇/7")
axes[2].set_title("Var/mean of pickups per cell"); axes[2].legend(fontsize=7)
fig.tight_layout(); plt.show()

### Does the $M_t/G/\infty$ model get the mean right?

Predicted mean cars on rent per (station, category, weekday) from the pickup rates and the pooled length survival of
section 3: $\hat E[T_{\mathrm{wd}}] = \sum_{j=0}^{9}\hat\lambda_{s,k,\mathrm{wd}-j}\,\hat S_{\text{type},k}(j+1)$, against the observed mean.

In [ ]:
lam = P.groupby(["station_id", "category", "weekday"], observed=True)["pickups"].mean()
SURV, PMF = {}, {}
for (t, k), g in hw.groupby(["station_type", "category"], observed=True):
    pmf = g["length"].value_counts(normalize=True).reindex(LS, fill_value=0).to_numpy()
    PMF[(t, k)] = pmf / pmf.sum()
    SURV[(t, k)] = pmf[::-1].cumsum()[::-1]                     # SURV[j] = P(L >= j + 1)

rows = []
for s in STATIONS:
    for k in CATS:
        S_ = SURV[(ST.loc[s, "station_type"], k)]
        for wi, w in enumerate(WEEKDAYS):
            rows.append({"station_id": s, "category": k, "weekday": w,
                         "pred": sum(lam.loc[(s, k, WEEKDAYS[(wi - j) % 7])] * S_[j] for j in range(MAXLEN))})
pred = pd.DataFrame(rows).merge(
    T.groupby(["station_id", "category", "weekday"], observed=True)["on_rent"].mean().rename("obs").reset_index()
     .astype({"category": str, "weekday": str}), on=["station_id", "category", "weekday"])
rel = pred["obs"] / pred["pred"] - 1
fig, ax = plt.subplots(figsize=(5, 4.5))
for k in CATS:
    e = pred[pred["category"] == k]
    ax.scatter(e["pred"], e["obs"], s=10, alpha=0.7, color=CAT_COLOR[k], label=k)
m = pred[["pred", "obs"]].max().max() * 1.05
ax.plot([0, m], [0, m], color="#0b0b0b", lw=1); ax.set_xlabel("predicted mean T"); ax.set_ylabel("observed mean T")
ax.set_title(f"mean cars on rent: obs / pred − 1\nmean {rel.mean():+.3f}, sd {rel.std():.3f}"); ax.legend(fontsize=7)
fig.tight_layout(); plt.show()

### Simulated null: what would Poisson pickups + independent lengths give?

The pooled statistics above have no simple reference distribution for $T$ (only 6–7 values per cell, overlapping
rentals, means estimated from the same data). So we simulate the history window 200 times under the null:
- pickups $\sim$ Poisson($\hat\lambda_{s,k,\mathrm{wd}}$), independent across days and cells;
- each pickup gets an independent length from the pooled pmf of its station type × category;
- the same truncation (cars on rent counted only from the 10th day of the window).

For each run we compute the same statistics. If the observed value sits inside the simulated distribution, the
data are consistent with Poisson pickups and independent lengths.

In [ ]:
rng = np.random.default_rng(2026)
REPS = 200
n_days = len(dates)
wd_idx = np.array([WEEKDAYS.index(d.day_name()[:3]) for d in dates])
full = np.asarray(dates >= FULL_FROM)
acc = {key: {"ratio": [], "var": [], "mean": []} for key in ["P", "T"]}
for s in STATIONS:
    for k in CATS:
        lam_d = lam.loc[(s, k)].reindex(WEEKDAYS).to_numpy()[wd_idx]
        N = rng.poisson(lam_d, size=(REPS, n_days))
        M = rng.multinomial(N, PMF[(ST.loc[s, "station_type"], k)])        # (REPS, n_days, MAXLEN)
        Tsim = np.zeros((REPS, n_days))
        for l in LS:                                                       # length-l pickups on d-l+1 .. d
            c = np.cumsum(M[:, :, l - 1], axis=1)
            lagged = np.zeros_like(c); lagged[:, l:] = c[:, :-l]
            Tsim += c - lagged
        for key, X, mask in [("P", N, np.ones(n_days, bool)), ("T", Tsim, full)]:
            for w in range(7):
                sel = mask & (wd_idx == w)
                mu_, var_ = X[:, sel].mean(1), X[:, sel].var(1, ddof=1)
                acc[key]["mean"].append(mu_); acc[key]["var"].append(var_)
                acc[key]["ratio"].append(np.where(mu_ > 0, var_ / np.where(mu_ > 0, mu_, 1), np.nan))

null = {}
for key in ["P", "T"]:
    R = np.stack(acc[key]["ratio"], 1); V = np.stack(acc[key]["var"], 1); Mu = np.stack(acc[key]["mean"], 1)
    null[key] = pd.DataFrame({"median": np.nanmedian(R, 1), "mean": np.nanmean(R, 1), "phi": V.sum(1) / Mu.sum(1) - 1})

fig, axes = plt.subplots(2, 3, figsize=(15, 6))
rows = []
for i, (key, lab) in enumerate([("P", "pickups"), ("T", "cars on rent")]):
    for j, stat in enumerate(["median", "mean", "phi"]):
        ax = axes[i, j]
        ax.hist(null[key][stat], bins=30, color="#9a9993", edgecolor="white", linewidth=0.3)
        ax.axvline(obs[key][stat], color="#eb6834", lw=2)
        p = (null[key][stat] >= obs[key][stat]).mean()
        ax.set_title(f"{lab}: {stat} Var/mean" if stat != "phi" else f"{lab}: pooled excess φ", fontsize=9)
        rows.append({"series": lab, "statistic": stat, "observed": obs[key][stat],
                     "null 5 %": null[key][stat].quantile(0.05), "null median": null[key][stat].median(),
                     "null 95 %": null[key][stat].quantile(0.95), "P(null ≥ obs)": p})
fig.suptitle("Null distribution (grey, 200 simulated histories) vs. observed (orange)", y=1.0)
fig.tight_layout(); plt.show()
display(pd.DataFrame(rows).round(3))

### Where does the overdispersion come from?

Lengths are independent (section 3), so the excess must sit in the pickups. Three diagnostics:
1. **Persistence:** $\phi_{\text{week}}/\phi_{\text{day}}$, where $\phi_{\text{week}}$ is the pooled excess of weekly
   pickup totals per cell. With i.i.d. daily rate shocks the ratio is 1; with one shock shared by all 7 days of a week it is 7.
2. **Lag profile** of standardized daily pickups $(P-\mu_{s,k,\mathrm{wd}})/\sqrt{\mu}$ within a cell, plus lag-1 split by
   weekday pair. If the shock is per **calendar** week, Sun → Mon has no correlation and the other pairs do.
   Note: centering on the mean of 8 same-weekday values alone gives about −1/7 at lags 7 and 14.
3. **Carry-over:** correlation of a cell's weekly level with the following weeks, and how well recent weeks predict the
   last one. If the level carries over, recent weeks should get more weight than the 8-week mean.

Then the size of the shock: with $P_{d}\mid G\sim\mathrm{Poisson}(G\,\mu_d)$ and one factor $G$ per (station, category,
calendar week), $E[G]=1$, $\mathrm{Var}[G]=v$, the weekly total $W$ has $\mathrm{Var}[W]=E[W]+v\,E[W]^2$, so
$\hat v=\sum(\mathrm{Var}\,W-\bar W)/\sum \bar W^2$ over cells.

In [ ]:
P["wd_i"] = P["date"].dt.day_of_week
P["wk"] = (P["date"] - H0).dt.days // 7
P["type"] = P["station_id"].map(ST["station_type"])

def phi_day(df):
    g = df.groupby(["station_id", "category", "wd_i"], observed=True)["pickups"].agg(["mean", "var"])
    return g["var"].sum() / g["mean"].sum() - 1

def phi_week(df):
    g = df.groupby(["station_id", "category", "wk"], observed=True)["pickups"].sum().groupby(level=[0, 1], observed=True).agg(["mean", "var"])
    return g["var"].sum() / g["mean"].sum() - 1

rows = [{"group": "all", "phi_day": phi_day(P), "phi_week": phi_week(P)}]
rows += [{"group": t, "phi_day": phi_day(g), "phi_week": phi_week(g)} for t, g in P.groupby("type")]
rows += [{"group": k, "phi_day": phi_day(g), "phi_week": phi_week(g)} for k, g in P.groupby("category", observed=True)]
pers = pd.DataFrame(rows); pers["week / day"] = pers["phi_week"] / pers["phi_day"]
print("Persistence (1 = i.i.d. daily shocks, 7 = one shock per week):")
display(pers.round(3))

# Lag profile of standardized daily pickups
mu_ = P.groupby(["station_id", "category", "wd_i"], observed=True)["pickups"].transform("mean")
P["zst"] = (P["pickups"] - mu_) / np.sqrt(mu_.where(mu_ > 0, 1))
Z = P.pivot_table(index=["station_id", "category"], columns="date", values="zst", observed=True)
lags = list(range(1, 15))
lag_c = [np.corrcoef(Z.iloc[:, :-l].to_numpy().ravel(), Z.iloc[:, l:].to_numpy().ravel())[0, 1] for l in lags]
pair_c = {}
for wd0, lab in enumerate(["Mon→Tue", "Tue→Wed", "Wed→Thu", "Thu→Fri", "Fri→Sat", "Sat→Sun", "Sun→Mon"]):
    cols = [j for j in range(len(dates) - 1) if dates[j].day_of_week == wd0]
    a = np.concatenate([Z.iloc[:, j].to_numpy() for j in cols]); b = np.concatenate([Z.iloc[:, j + 1].to_numpy() for j in cols])
    pair_c[lab] = np.corrcoef(a, b)[0, 1]

# Carry-over between weeks
Wk = P.groupby(["station_id", "category", "wk"], observed=True)["pickups"].sum().unstack()
R = Wk.div(Wk.mean(axis=1), axis=0).to_numpy() - 1
carry = {l: np.corrcoef(R[:, :-l].ravel(), R[:, l:].ravel())[0, 1] for l in [1, 2, 3]}
mse = {"mean of weeks 1–7": ((Wk[7] - Wk.loc[:, 0:6].mean(axis=1)) ** 2).mean(),
       "mean of weeks 6–7": ((Wk[7] - Wk.loc[:, 5:6].mean(axis=1)) ** 2).mean(),
       "week 7 only": ((Wk[7] - Wk[6]) ** 2).mean()}

# Size of the weekly shock
mW, vW = Wk.mean(axis=1), Wk.var(axis=1)
v_hat = (vW - mW).sum() / (mW ** 2).sum()

fig, axes = plt.subplots(1, 3, figsize=(17, 3.6))
ax = axes[0]
ax.bar(lags, lag_c, color=["#2a78d6" if l < 7 else "#9a9993" for l in lags])
ax.axhline(0, color="#52514e", lw=0.8); ax.axhline(-1 / 7, color="#eb6834", lw=1, ls="--", label="−1/7 (centering only)")
ax.set_xticks(lags); ax.set_xlabel("lag (days)"); ax.set_title("autocorrelation of standardized daily pickups"); ax.legend(fontsize=7)
ax = axes[1]
ax.bar(list(pair_c), list(pair_c.values()), color=["#9a9993" if k == "Sun→Mon" else "#2a78d6" for k in pair_c])
ax.axhline(0, color="#52514e", lw=0.8); ax.set_title("lag-1 correlation by weekday pair"); ax.tick_params(axis="x", rotation=30)
ax = axes[2]
ax.bar([f"lag {l} wk" for l in carry], list(carry.values()), color="#2a78d6")
ax.axhline(-1 / 7, color="#eb6834", lw=1, ls="--", label="−1/7 (centering only)"); ax.axhline(0, color="#52514e", lw=0.8)
ax.set_title("weekly level: corr with later weeks of the same cell"); ax.legend(fontsize=7)
fig.tight_layout(); plt.show()

print("Predict week 8 pickups per cell (MSE):", {k: round(v, 1) for k, v in mse.items()})
print(f"Weekly shock: v̂ = Var[G] = {v_hat:.4f}, i.e. sd of the weekly factor ≈ {np.sqrt(v_hat):.3f}")

### Shape of the cars-on-rent distribution, pooled

Standardized $(T - \mu_{s,k,\mathrm{wd}})/\sqrt{\mu_{s,k,\mathrm{wd}}}$ pooled over all cells, against $N(0,1)$.
Under Poisson its sd is about 1 (a little less, since $\mu$ is estimated from the same 6–7 values).

In [ ]:
mu = T.groupby(["station_id", "category", "weekday"], observed=True)["on_rent"].transform("mean")
zz = ((T["on_rent"] - mu) / np.sqrt(mu.where(mu > 0))).dropna()
nbar = T.groupby(["station_id", "category", "weekday"], observed=True).size().mean()
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
for ax, t in zip(axes, ["airport", "downtown"]):
    v = zz[T.loc[zz.index, "station_type"] == t]
    ax.hist(v, bins=50, density=True, color=TYPE_COLOR[t], edgecolor="white", linewidth=0.3)
    xg = np.linspace(-4, 4, 200)
    ax.plot(xg, np.exp(-xg ** 2 / 2) / np.sqrt(2 * np.pi), color="#0b0b0b", lw=1.2)
    ax.set_title(f"{t}: sd = {v.std():.2f} (Poisson ≈ {np.sqrt((nbar - 1) / nbar):.2f})")
fig.tight_layout(); plt.show()

## 5. Weekday pattern and trend

**Weekday index** = mean per weekday / overall mean, per station (lines coloured by station type). If the lines
of one type lie on top of each other, a shared weekday profile per station type × category is enough
(fewer parameters than 10 × 4 × 7 means of 8 values each).

In [ ]:
fig, axes = plt.subplots(2, len(CATS), figsize=(16, 6), sharey="row")
for row, (df, col, lab) in enumerate([(P, "pickups", "pickups"), (T, "on_rent", "cars on rent")]):
    m = df.groupby(["station_id", "category", "weekday"], observed=True)[col].mean()
    idx = m / m.groupby(["station_id", "category"], observed=True).transform("mean")
    for j, k in enumerate(CATS):
        ax = axes[row, j]
        for s in STATIONS:
            ax.plot(range(7), idx.loc[(s, k)].values, color=TYPE_COLOR[ST.loc[s, "station_type"]], lw=1, alpha=0.75)
        ax.axhline(1, color="#52514e", lw=0.8)
        ax.set_xticks(range(7), WEEKDAYS)
        ax.set_title(f"{k}: {lab}" if row == 0 else f"{k}: {lab}")
    axes[row, 0].set_ylabel("weekday index")
axes[0, 0].plot([], [], color=TYPE_COLOR["airport"], label="airport"); axes[0, 0].plot([], [], color=TYPE_COLOR["downtown"], label="downtown")
axes[0, 0].legend(fontsize=7)
fig.tight_layout(); plt.show()

**Trend.** Weekly pickups per station type × category, indexed to the 8-week mean. A trend would mean the
plain 8-week mean under-/overshoots the horizon.

In [ ]:
wk = P.groupby(["station_type", "category", "week"], observed=True)["pickups"].sum()
wk_idx = wk / wk.groupby(["station_type", "category"], observed=True).transform("mean")
fig, axes = plt.subplots(1, 2, figsize=(12, 3.4), sharey=True)
for ax, t in zip(axes, ["airport", "downtown"]):
    for k in CATS:
        e = wk_idx.loc[(t, k)]
        ax.plot(e.index, e.values, marker="o", ms=4, lw=2, color=CAT_COLOR[k], label=k)
    ax.axhline(1, color="#52514e", lw=0.8); ax.set_xlabel("history week"); ax.set_title(f"{t}: weekly pickups / mean")
axes[0].legend(fontsize=7); fig.tight_layout(); plt.show()
# Linear trend per station type x category, in % of the mean per week
tr = (wk_idx.reset_index().groupby(["station_type", "category"], observed=True)
        .apply(lambda g: 100 * np.polyfit(g["week"], g["pickups"], 1)[0], include_groups=False).rename("trend %/week"))
display(tr.unstack().round(2))

## 6. Correlations between cells

Residuals $e = P - \bar P_{s,k,\mathrm{wd}}$ of daily pickups. Positive correlation **between categories at one
station** means common shocks (busy days), which matter for upgrades: the spare cars of the higher category are
scarce exactly when the lower one is short. Positive correlation **between stations** means network-wide shocks,
which a scenario generator with independent cells would miss (and transfers hedge less than we think).
Between stations we use pickups: cars on rent are strongly autocorrelated over days, so their sample correlations
over 47 dates are much noisier than they look.

In [ ]:
P["resid"] = P["pickups"] - P.groupby(["station_id", "category", "weekday"], observed=True)["pickups"].transform("mean")
T["resid"] = T["on_rent"] - T.groupby(["station_id", "category", "weekday"], observed=True)["on_rent"].transform("mean")

def avg_corr(df, by, across):
    """Mean over `by` groups of the correlation matrix between the `across` series (over dates)."""
    mats = [g.pivot(index="date", columns=across, values="resid").corr() for _, g in df.groupby(by, observed=True)]
    return sum(mats) / len(mats)

fig, axes = plt.subplots(1, 3, figsize=(16, 4.4), gridspec_kw={"width_ratios": [1, 1, 1.6]})
for ax, (df, lab) in zip(axes[:2], [(P, "pickups"), (T, "cars on rent")]):
    C = avg_corr(df, "station_id", "category").reindex(index=CATS, columns=CATS)
    ax.imshow(C.values, cmap="RdBu_r", vmin=-0.5, vmax=0.5); ax.grid(False)
    ax.set_xticks(range(4), CATS, rotation=30); ax.set_yticks(range(4), CATS)
    for i in range(4):
        for j in range(4):
            ax.text(j, i, f"{C.values[i, j]:+.2f}", ha="center", va="center", fontsize=8)
    ax.set_title(f"{lab}: corr between categories\n(same station, mean over stations)")
C = avg_corr(P, "category", "station_id").reindex(index=STATIONS, columns=STATIONS)
im = axes[2].imshow(C.values, cmap="RdBu_r", vmin=-0.5, vmax=0.5); axes[2].grid(False)
axes[2].set_xticks(range(10), STATIONS); axes[2].set_yticks(range(10), STATIONS)
for i in range(10):
    for j in range(10):
        axes[2].text(j, i, f"{C.values[i, j]:+.2f}", ha="center", va="center", fontsize=6)
axes[2].set_title("pickups: corr between stations\n(same category, mean over categories; 56 days, so |ρ| < 0.27 is noise)")
fig.colorbar(im, ax=axes[2], shrink=0.8)
fig.tight_layout(); plt.show()

off = C.values[~np.eye(10, dtype=bool)]
print(f"between stations (pickups): mean off-diagonal corr = {off.mean():+.3f}")
tot = P.groupby("date")["resid"].sum()
print(f"network pickups per day: Var(sum of residuals) / sum of cell variances = "
      f"{tot.var() / P.groupby(['station_id', 'category'], observed=True)['resid'].var().sum():.2f} (1 = independent cells)")

## 7. Empirical demand to come (pseudo-snapshots, route B)

The quantity the scorer uses is $D_{s,k,d}$: cars on rent on horizon day $d$ from rentals **booked on or after day 1**.
We can observe exactly this in the history: take every Monday $t_0$ in the window as a pseudo "day 1" and count, for
$d = 1..14$, cars on rent on $t_0 + d - 1$ from rentals with booking date $\ge t_0$. All those rentals are picked up
on or after $t_0$, so there is no truncation; only the last snapshots run past the window (8 values for $d \le 7$, 7 for $d \ge 8$).

The same snapshots give the pseudo-**committed** $B$ (rentals booked before $t_0$), so we can see whether $B$ and
$D$ move together.

In [ ]:
mondays = [t for t in dates if t.day_of_week == 0]
ex = explode_days(hw)
ex = ex[ex["date"] <= H1]
snaps = []
for t0 in mondays:
    e = ex[(ex["date"] >= t0) & (ex["date"] <= t0 + pd.Timedelta(days=13))].copy()
    e["to_come"] = e["booking_date"] >= t0
    e["day"] = (e["date"] - t0).dt.days + 1
    g = e.groupby(["pickup_station_id", "category", "day", "to_come"], observed=True).size().unstack(fill_value=0)
    days = range(1, min(14, (H1 - t0).days + 1) + 1)
    idx = pd.MultiIndex.from_product([STATIONS, CATS, days], names=["station_id", "category", "day"])
    g = g.rename_axis(index=["station_id", "category", "day"]).reindex(idx, fill_value=0)
    g = g.rename(columns={True: "D", False: "B"}).reset_index()
    g["t0"] = t0
    if t0 < FULL_FROM:          # rentals picked up before the window are missing, so B is truncated
        g["B"] = np.nan
    snaps.append(g)
snap = pd.concat(snaps, ignore_index=True)
snap["category"] = pd.Categorical(snap["category"], CATS, ordered=True)
print(f"{len(mondays)} pseudo-snapshots:", [t.date().isoformat() for t in mondays])
print(f"B (pseudo-committed) is complete only for snapshots from {FULL_FROM.date()} on; D is complete for all.")

fig, axes = grid_axes(len(STATIONS), len(CATS), w=3.7, h=1.8)
for i, s in enumerate(STATIONS):
    for j, k in enumerate(CATS):
        ax = axes[i, j]
        g = snap[(snap["station_id"] == s) & (snap["category"] == k)]
        vals = [g.loc[g["day"] == d, "D"].to_numpy() for d in range(1, 15)]
        ax.boxplot(vals, positions=range(1, 15), widths=0.6, showfliers=False, whis=(0, 100),
                   medianprops={"color": CAT_COLOR[k], "lw": 1.5},
                   boxprops={"color": "#9a9993"}, whiskerprops={"color": "#9a9993"}, capprops={"color": "#9a9993"})
        ax.plot(range(1, 15), [v.mean() for v in vals], color=CAT_COLOR[k], lw=1.5)
        ax.set_ylim(bottom=0)
        ax.set_xticks(range(1, 15), [str(d) if d % 2 else "" for d in range(1, 15)] if i == len(STATIONS) - 1 else [""] * 14)
label_grid(axes)
for ax in axes[-1]:
    ax.set_xlabel("horizon day (1 = Mon)")
fig.suptitle("Empirical demand to come D per horizon day (boxes = Monday pseudo-snapshots, line = mean)", y=1.0, fontsize=12)
fig.tight_layout(); plt.show()

### Pseudo-committed $B$, and does $B$ predict $D$?

Left: network total of pseudo-committed $B$ per horizon day, one line per history snapshot.
Right: per (station, category, day ≤ 7), the residual of $D$ against the residual of $B$ (both relative to their cell
mean over snapshots). A positive slope means "more booked than usual ⇒ more still to come", the
correlation the naive "mean − committed" forecast gets wrong (see [[Project Plan]], §2).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 3.8))
ax = axes[0]
for k in CATS:
    tot = snap[(snap["category"] == k) & snap["B"].notna()].groupby(["t0", "day"])["B"].sum().unstack("t0")
    ax.plot(tot.index, tot.values, color=CAT_COLOR[k], lw=1, alpha=0.7)
    ax.plot([], [], color=CAT_COLOR[k], lw=1.5, label=k)
ax.set_xlabel("horizon day"); ax.set_ylabel("committed car-days, all stations"); ax.legend(fontsize=7)
ax.set_title("pseudo-committed B per history snapshot")

ax = axes[1]
s7 = snap[(snap["day"] <= 7) & snap["B"].notna()].copy()
for c in ["B", "D"]:
    s7[c + "_res"] = s7[c] - s7.groupby(["station_id", "category", "day"], observed=True)[c].transform("mean")
for k in CATS:
    e = s7[s7["category"] == k]
    ax.scatter(e["B_res"], e["D_res"], s=6, alpha=0.5, color=CAT_COLOR[k], label=k)
beta = np.polyfit(s7["B_res"], s7["D_res"], 1)[0]
rho = np.corrcoef(s7["B_res"], s7["D_res"])[0, 1]
xg = np.array([s7["B_res"].min(), s7["B_res"].max()]); ax.plot(xg, beta * xg, color="#0b0b0b", lw=1.2)
ax.set_xlabel("B − mean(B)"); ax.set_ylabel("D − mean(D)")
ax.set_title(f"residual D vs. residual B, days 1–7: ρ = {rho:+.2f}, slope β = {beta:+.2f}")
ax.legend(fontsize=7, markerscale=2)
fig.tight_layout(); plt.show()

## 8. Takeaways (2026-11-13)

**Rental length (section 3)**
- One **truncated geometric** fits everything: $\pi(l)\propto q^{l-1}$, $l=1..10$, with $q\approx0.69$ (airport) /
  $0.70$ (downtown); mean ≈ 3.0 car-days.
- **Pooling is fine:** there is no difference between station type × category groups (χ² p = 0.54), between categories (p = 0.72) or between
  station types (p = 0.37). There is weak evidence for station effects inside a group (p ≈ 0.02–0.04 over 224–273 df), too small to matter.
- **Independent:** length doesn't depend on pickup weekday, history week or how busy the day was (ρ = 0.00), and there is
  no day-level clustering (Var(z) = 1.02). Spearman(length, lead) ≈ 0. So the $M_t/G/\infty$ machinery applies, and
  the convolution $\lambda * S$ reproduces the mean cars on rent to within 0.5 % on average (sd 7 % per cell).

**Dispersion (section 4): not Poisson, and we know why**
- Pickups and cars on rent are **overdispersed**. All pooled statistics are far outside the simulated Poisson null
  (p < 1/200): pooled excess φ = 0.34 for pickups, 0.61 for cars on rent, against ≈ 0 ± 0.08 under the null.
- The excess is **one multiplicative shock per (station, category, calendar week Mon–Sun)**:
  - $\phi_{\text{week}}/\phi_{\text{day}} \approx 5.8$ (7 = fully shared within the week);
  - daily residuals are correlated within a week (+0.07 to +0.32) and **not from Sunday to Monday** (−0.01);
  - the autocorrelation is gone by lag 5.
- **Independent across weeks:** week-to-week correlation of a cell's level ≈ −1/7, which centering alone produces.
  The 8-week mean predicts the last week better than recent weeks do (MSE 287 vs. 358 for the last two weeks).
- **Independent across cells:** ≈ 0 correlation between stations and between categories (section 6).
- **Size:** $\hat v = \mathrm{Var}[G] \approx 0.017$, i.e. the weekly demand level of a cell varies by about **±13 %**
  around its mean.
- **Model for demand:** $P_{s,k,d}\mid G \sim \mathrm{Poisson}(G_{s,k,w(d)}\,\lambda_{s,k,\mathrm{wd}(d)})$ with
  $G\sim\mathrm{Gamma}$(mean 1, var 0.017), i.i.d. per cell and calendar week; lengths i.i.d. truncated geometric. Within a week this is
  NegBin. The horizon is exactly two calendar weeks, so each cell gets two independent factors.

**Weekday and trend (section 5)**
- **Strong weekday pattern that depends on the station type:** downtown peaks Fri–Sun (Sat pickups ≈ 2.2× the mean), airports Mon and Thu.
  Within a type the stations' weekday profiles almost coincide, in every category.
- **No trend.** The apparent decline in downtown economy (−2.6 %/week fitted) comes from a few high early weeks at
  individual stations, which is the weekly shock above, not a trend. Use the plain 8-week mean.

**Demand to come (section 7)**
- A ramp over the first ~5 horizon days (near days are mostly booked already), then the weekday pattern.
- The residual correlation of $D$ with the pseudo-committed $B$ is +0.17 (slope +0.14). The weekly shock explains it: a
  busy week shows up both in what is already booked and in what is still to come. So the committed book carries
  information about the week-1 factor $G_{s,k,1}$ of each cell (not about week 2).

**Implications for the forecast and the model**
- **Expected demand:** weekday means of 8 weeks (or level × type profile), convolved with the pooled length survival;
  optionally update the week-1 level from the committed book.
- **Scenarios / expected revenue curves:** draw the weekly Gamma factor per cell. Pure Poisson would understate the
  variance of a cell's cars on rent on a day by a factor of $1+v\mu$ (×1.9 at $\mu = 50$), and
  with it the concavity of $E[\text{revenue}](a)$.